In [4]:


import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.width", 120)
np.random.seed(42)


print("BUILDING A MESSY SYNTHETIC DATASET")


n = 300
df = pd.DataFrame({
    "age": np.random.normal(40, 12, n).round(1),
    "income": np.random.exponential(50000, n).round(0),
    "education": np.random.choice(["highschool", "bachelor", "master", "phd"], n,
                                   p=[0.4, 0.35, 0.2, 0.05]),
    "city": np.random.choice(["Dhaka", "Chattogram", "Sylhet", "Khulna", "Rajshahi"], n),
    "target": np.random.choice([0, 1], n, p=[0.7, 0.3])
})

# Inject missing values
missing_idx = np.random.choice(df.index, size=30, replace=False)
df.loc[missing_idx, "income"] = np.nan
missing_idx2 = np.random.choice(df.index, size=15, replace=False)
df.loc[missing_idx2, "education"] = np.nan

# Inject outliers
outlier_idx = np.random.choice(df.index, size=5, replace=False)
df.loc[outlier_idx, "income"] = df["income"].max() * 8

print(df.head())
print("\nShape:", df.shape)


print("1) MISSING VALUES")

print("\nMissing counts:\n", df.isnull().sum())
print("\nMissing %:\n", (df.isnull().mean() * 100).round(2))

# Missingness indicator (often more useful than the imputed value itself)
df["income_was_missing"] = df["income"].isnull().astype(int)


print("2) OUTLIERS (IQR method)")

Q1, Q3 = df["income"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = df[(df["income"] < lower) | (df["income"] > upper)]
print(f"IQR bounds: [{lower:.0f}, {upper:.0f}]")
print(f"Outliers detected: {len(outliers)}")
print(outliers[["income"]])

# Winsorize (cap) instead of dropping — keeps the row, tames the extreme value
df["income_capped"] = df["income"].clip(lower=lower, upper=upper)
print("\nIncome before/after capping (outlier rows):")
print(df.loc[outliers.index, ["income", "income_capped"]])


print("3) ENCODING CATEGORICAL VARIABLES")


# Ordinal encoding — education has a natural order
edu_order = ["highschool", "bachelor", "master", "phd"]
ordinal_enc = OrdinalEncoder(categories=[edu_order], handle_unknown="use_encoded_value", unknown_value=-1)
edu_filled = df["education"].fillna("highschool")  # simple fill for this quick demo
edu_encoded = ordinal_enc.fit_transform(edu_filled.values.reshape(-1, 1))
print("Ordinal encoding sample (education):")
print(pd.DataFrame({"education": edu_filled.head(8).values, "encoded": edu_encoded[:8].ravel()}))

# One-hot encoding — city has no natural order
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
city_encoded = ohe.fit_transform(df[["city"]])
print("\nOne-hot encoded city columns:", ohe.get_feature_names_out())


print("4) STANDARDIZATION vs 5) NORMALIZATION")


age_vals = df[["age"]].values

standard_scaler = StandardScaler()
age_standardized = standard_scaler.fit_transform(age_vals)
print(f"Age — original mean/std : {age_vals.mean():.2f} / {age_vals.std():.2f}")
print(f"Age — standardized mean/std : {age_standardized.mean():.2f} / {age_standardized.std():.2f}")

minmax_scaler = MinMaxScaler()
age_normalized = minmax_scaler.fit_transform(age_vals)
print(f"Age — normalized (min-max) range : [{age_normalized.min():.2f}, {age_normalized.max():.2f}]")


print("6) FEATURE TRANSFORMATION (fixing skew)")

print(f"Income skewness before transform: {df['income_capped'].skew():.3f}")
pt = PowerTransformer(method="yeo-johnson")
income_transformed = pt.fit_transform(df[["income_capped"]].fillna(df["income_capped"].median()))
print(f"Income skewness after Yeo-Johnson: {pd.Series(income_transformed.ravel()).skew():.3f}")


print("7) ColumnTransformer + 8) Pipeline — putting it all together")


X = df[["age", "income", "education", "city"]]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

numeric_features = ["age", "income"]
categorical_features = ["education", "city"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42))
])

full_pipeline.fit(X_train, y_train)
test_accuracy = full_pipeline.score(X_test, y_test)
print(f"Test accuracy of full pipeline: {test_accuracy:.4f}")

# Cross-validate the WHOLE pipeline — imputers/scalers/encoders refit correctly per fold
cv_scores = cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring="accuracy")
print(f"5-fold CV accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

print("\nTransformed feature names after preprocessing:")
print(preprocessor.fit(X_train).get_feature_names_out())

print("\nDone.")


BUILDING A MESSY SYNTHETIC DATASET
    age    income   education        city  target
0  46.0   16567.0    bachelor      Sylhet       0
1  38.3    9767.0  highschool  Chattogram       0
2  47.8   69438.0      master       Dhaka       1
3  58.3   82210.0  highschool      Sylhet       0
4  37.2  232850.0  highschool      Sylhet       0

Shape: (300, 5)
1) MISSING VALUES

Missing counts:
 age           0
income       30
education    15
city          0
target        0
dtype: int64

Missing %:
 age           0.0
income       10.0
education     5.0
city          0.0
target        0.0
dtype: float64
2) OUTLIERS (IQR method)
IQR bounds: [-82094, 174525]
Outliers detected: 18
        income
4     232850.0
33   3268976.0
53   3268976.0
55    215773.0
60    213439.0
85    175126.0
89    247841.0
112   183248.0
113   214193.0
132   218214.0
136   175185.0
145   408622.0
146   284744.0
241  3268976.0
275   215310.0
276  3268976.0
289   201932.0
299  3268976.0

Income before/after capping (outlier ro